# Project 7 — RAG Document Chatbot

## Overview
**Retrieval-Augmented Generation (RAG)** combines a retrieval system with an LLM to answer questions grounded in a document corpus. This project builds a complete RAG pipeline from scratch.

```
Documents → Chunking → Embeddings → Vector Store
                                          ↑
User Query → Embed → Retrieve top-k → LLM → Answer
```

## 1. Theory — RAG

### Why RAG?
LLMs have a fixed training knowledge cutoff and cannot access private documents. Two solutions:
- **Fine-tuning**: expensive, hard to update, risks forgetting
- **RAG**: inject relevant documents at inference time — cheap, always up-to-date, transparent

### Indexing Phase (Offline)
1. **Chunk** documents into smaller pieces (semantic units)
2. **Embed** each chunk into a vector: $\mathbf{e} = f(\text{chunk})$
3. **Store** $(\text{chunk}, \mathbf{e})$ in a vector database

### Query Phase (Online)
1. Embed query: $\mathbf{q} = f(\text{query})$
2. Retrieve top-$k$ chunks by cosine similarity:
$$\text{sim}(\mathbf{q}, \mathbf{e}) = \frac{\mathbf{q} \cdot \mathbf{e}}{\|\mathbf{q}\| \|\mathbf{e}\|}$$
3. Construct prompt: `Context: {retrieved} | Question: {query}`
4. LLM generates grounded answer

### Chunking Strategies
| Strategy | Description | Best for |
|---|---|---|
| Fixed-size | Split every N characters with M overlap | Simple, fast |
| Sentence | Split at sentence boundaries | Better coherence |
| Semantic | Embed and split where meaning changes | Best quality |

### Embeddings
Sentence transformers like `all-MiniLM-L6-v2` produce 384-dimensional dense vectors that capture semantic meaning. Similar texts have high cosine similarity.

In [ ]:
# Install required packages if not present
# !pip install sentence-transformers

import numpy as np
from dataclasses import dataclass
from typing import List, Tuple
import textwrap

print('Imports done.')

## 2. Document Corpus
We define a set of documents as Python strings — no external files needed.

In [ ]:
DOCUMENTS = [
    {
        'title': 'Python Programming Language',
        'content': '''Python is a high-level, general-purpose programming language. Its design philosophy emphasises code readability with the use of significant indentation. Python is dynamically typed and garbage-collected. It supports multiple programming paradigms, including structured, object-oriented and functional programming. Python was created by Guido van Rossum and first released in 1991. Python consistently ranks as one of the most popular programming languages. Python's large standard library is often cited as one of its greatest strengths. The language uses whitespace indentation rather than curly brackets to delimit blocks of code. Python uses dynamic typing and a combination of reference counting and cycle-detecting garbage collection for memory management.'''
    },
    {
        'title': 'Machine Learning Overview',
        'content': '''Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalise to unseen data, and thus perform tasks without explicit instructions. Recently, AI has been used to create large language models, which are trained on vast amounts of text data. Machine learning algorithms build a model based on sample data, known as training data, in order to make predictions or decisions without being explicitly programmed to do so. Machine learning is used in a wide variety of applications, such as in medicine, email filtering, speech recognition, agriculture, and computer vision, where it is difficult or unfeasible to develop conventional algorithms to perform the needed tasks.'''
    },
    {
        'title': 'Deep Learning and Neural Networks',
        'content': '''Deep learning is part of a broader family of machine learning methods, which is based on artificial neural networks with representation learning. Learning can be supervised, semi-supervised or unsupervised. Deep learning architectures such as deep neural networks, recurrent neural networks, convolutional neural networks, and transformers have been applied to fields including computer vision, speech recognition, natural language processing, machine translation, bioinformatics, and drug design. Artificial neural networks were inspired by information processing and distributed communication nodes in biological systems. Backpropagation is the key algorithm for training neural networks. The "deep" in deep learning refers to multiple layers in the neural network.'''
    },
    {
        'title': 'Transformer Architecture',
        'content': '''A transformer is a deep learning architecture developed by researchers at Google and based on the multi-head attention mechanism, proposed in the 2017 paper "Attention Is All You Need". The transformer architecture consists of an encoder and a decoder, each made up of multiple layers. Each layer has two sublayers: a multi-head self-attention mechanism and a position-wise fully connected feed-forward network. Transformers have become the dominant architecture for NLP tasks. BERT and GPT are well-known transformer-based models. The attention mechanism allows the model to weigh the importance of different words in the input sequence. Positional encodings are added to give the model information about the position of tokens in the sequence.'''
    },
    {
        'title': 'Vector Databases',
        'content': '''A vector database is a database that stores data as high-dimensional vectors. Vector databases are designed to efficiently store and query high-dimensional vector data, making them ideal for AI applications that use embeddings. Popular vector databases include Pinecone, Weaviate, Qdrant, Chroma, and FAISS. FAISS (Facebook AI Similarity Search) is an open-source library for efficient similarity search and clustering of dense vectors. Vector databases support approximate nearest neighbor (ANN) search algorithms, such as HNSW and IVF, which trade a small amount of accuracy for much faster query times. The key operations in a vector database are indexing (adding vectors) and querying (finding similar vectors).'''
    },
    {
        'title': 'Large Language Models',
        'content': '''A large language model (LLM) is a type of artificial intelligence (AI) program that can recognize and generate text, among other tasks. LLMs are trained on huge sets of data. GPT-4, Claude, Gemini, and LLaMA are examples of large language models. LLMs use the transformer architecture and are trained to predict the next token given context. The training involves predicting masked tokens or next tokens on vast text datasets. LLMs can be fine-tuned for specific tasks using supervised learning or RLHF (Reinforcement Learning from Human Feedback). Prompting techniques like chain-of-thought and few-shot prompting can significantly improve LLM performance.'''
    },
    {
        'title': 'Retrieval Augmented Generation',
        'content': '''Retrieval-augmented generation (RAG) is a technique for enhancing the accuracy and reliability of generative AI models with facts fetched from external sources. RAG is a combination of a retrieval system and a generative model. The retrieval system fetches relevant documents from a corpus, and the generative model uses these documents to generate an answer. RAG was introduced in a 2020 paper by Facebook AI Research. RAG helps address the problem of LLM hallucinations by grounding responses in retrieved evidence. The key components are: a document store, an embedding model, a vector index, a retriever, and an LLM generator.'''
    },
]

print(f'Loaded {len(DOCUMENTS)} documents')
for doc in DOCUMENTS:
    print(f"  - {doc['title']} ({len(doc['content'])} chars)")

## 3. Text Chunking
We split documents into overlapping fixed-size chunks to improve retrieval granularity.

In [ ]:
def chunk_text(text: str, chunk_size: int = 200, overlap: int = 50) -> List[str]:
    """
    Split text into overlapping fixed-size character chunks.

    Args:
        text:       Input text
        chunk_size: Max characters per chunk
        overlap:    Character overlap between consecutive chunks

    Returns:
        List of text chunks
    """
    chunks = []
    start  = 0
    while start < len(text):
        end   = min(start + chunk_size, len(text))
        chunk = text[start:end].strip()
        if chunk:  # skip empty chunks
            chunks.append(chunk)
        if end == len(text):
            break
        start = end - overlap  # slide with overlap
    return chunks


# Chunk all documents and record metadata
all_chunks = []
for doc in DOCUMENTS:
    chunks = chunk_text(doc['content'], chunk_size=200, overlap=50)
    for i, chunk in enumerate(chunks):
        all_chunks.append({
            'text':   chunk,
            'title':  doc['title'],
            'chunk_id': i
        })

print(f'Total chunks: {len(all_chunks)}')
print('\nSample chunk:')
print(f"  Title: {all_chunks[0]['title']}")
print(f"  Text:  {all_chunks[0]['text'][:100]}...")

## 4. Embedding Generation
We use `sentence-transformers` to embed chunks and queries into a shared vector space.

In [ ]:
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2: fast, 384-dim, good quality
print('Loading sentence-transformer model...')
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

# Embed all chunks
texts   = [c['text'] for c in all_chunks]
print(f'Embedding {len(texts)} chunks...')
embeddings = embed_model.encode(texts, show_progress_bar=True, convert_to_numpy=True)

print(f'\nEmbedding matrix shape: {embeddings.shape}')
print(f'Embedding dimension:    {embeddings.shape[1]}')

## 5. Vector Store
A simple numpy-based vector store using cosine similarity.

In [ ]:
class VectorStore:
    """Simple in-memory vector store using cosine similarity."""

    def __init__(self):
        self.texts      = []
        self.metadata   = []
        self.embeddings = None  # (N, D) array

    def add(self, texts: List[str], embeddings: np.ndarray, metadata: List[dict] = None):
        """Add documents and their embeddings to the store."""
        self.texts.extend(texts)
        self.metadata.extend(metadata or [{} for _ in texts])

        if self.embeddings is None:
            self.embeddings = embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, embeddings])

        # L2-normalise for efficient cosine similarity via dot product
        norms = np.linalg.norm(self.embeddings, axis=1, keepdims=True)
        self.embeddings_norm = self.embeddings / (norms + 1e-10)

    def search(self, query_embedding: np.ndarray, k: int = 3) -> List[dict]:
        """
        Find top-k most similar documents.

        Returns list of dicts with text, metadata, and similarity score.
        """
        # Normalise query
        q_norm = query_embedding / (np.linalg.norm(query_embedding) + 1e-10)

        # Cosine similarities via dot product (since both are normalised)
        sims   = self.embeddings_norm @ q_norm  # (N,)

        # Get top-k indices (descending)
        top_k_idx = np.argsort(sims)[::-1][:k]

        results = []
        for idx in top_k_idx:
            results.append({
                'text':     self.texts[idx],
                'metadata': self.metadata[idx],
                'score':    float(sims[idx])
            })
        return results


# Build the vector store
vs = VectorStore()
vs.add(
    texts=[c['text'] for c in all_chunks],
    embeddings=embeddings,
    metadata=[{'title': c['title'], 'chunk_id': c['chunk_id']} for c in all_chunks]
)

print(f'Vector store built: {len(vs.texts)} documents indexed')

## 6. RAG Pipeline

In [ ]:
def retrieve(query: str, k: int = 3) -> List[dict]:
    """Embed query and retrieve top-k relevant chunks."""
    query_emb = embed_model.encode([query], convert_to_numpy=True)[0]
    return vs.search(query_emb, k=k)


def build_prompt(query: str, retrieved_chunks: List[dict]) -> str:
    """Build RAG prompt from query and retrieved context."""
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks):
        source = chunk['metadata'].get('title', 'Unknown')
        context_parts.append(f"[{i+1}] (Source: {source})\n{chunk['text']}")

    context = '\n\n'.join(context_parts)
    prompt  = f"""Answer the question using ONLY the context provided below. Cite sources by number [1], [2], etc.

Context:
{context}

Question: {query}

Answer:"""
    return prompt


def simple_answer(query: str, chunks: List[dict]) -> str:
    """
    Template-based answer generator (no API key needed).
    In production, replace this with an LLM API call.
    """
    # Combine retrieved text and generate a simple answer
    combined = ' '.join([c['text'] for c in chunks])
    # Find sentences most relevant to the query keywords
    sentences  = combined.replace('\n', ' ').split('. ')
    query_words = set(query.lower().split())

    # Score sentences by keyword overlap
    scored = []
    for sent in sentences:
        words = set(sent.lower().split())
        score = len(words & query_words)
        scored.append((score, sent))

    scored.sort(reverse=True)
    top_sentences = [s for _, s in scored[:2] if s.strip()]

    sources = list({c['metadata'].get('title', '?') for c in chunks})
    answer  = '. '.join(top_sentences) + '.'
    answer += f"\n\n[Sources: {', '.join(sources)}]"
    return answer


def rag_answer(query: str, k: int = 3) -> dict:
    """Full RAG pipeline: retrieve + generate."""
    chunks = retrieve(query, k=k)
    prompt = build_prompt(query, chunks)
    answer = simple_answer(query, chunks)
    return {
        'query':   query,
        'chunks':  chunks,
        'prompt':  prompt,
        'answer':  answer
    }


print('RAG pipeline ready.')

## 7. Demo Questions

In [ ]:
QUESTIONS = [
    'What is Python and when was it created?',
    'How does the attention mechanism work in transformers?',
    'What is RAG and how does it help with LLM hallucinations?',
    'What are vector databases used for?',
    'How does deep learning differ from traditional machine learning?',
]

for q in QUESTIONS:
    result = rag_answer(q, k=3)
    print(f'{'='*60}')
    print(f'QUESTION: {q}')
    print(f'\nRETRIEVED ({len(result["chunks"])} chunks):')
    for chunk in result['chunks']:
        print(f"  [{chunk['score']:.3f}] {chunk['metadata']['title']}: {chunk['text'][:60]}...")
    print(f'\nANSWER:\n{result["answer"]}')
    print()

## 8. Retrieval Evaluation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='whitegrid')

# Visualise similarity scores for all test queries
fig, axes = plt.subplots(1, len(QUESTIONS), figsize=(16, 4))

for ax, q in zip(axes, QUESTIONS):
    chunks = retrieve(q, k=len(all_chunks))  # get all scores
    top5   = chunks[:5]
    labels = [c['metadata']['title'][:15] + f"\n(c{c['metadata']['chunk_id']})" for c in top5]
    scores = [c['score'] for c in top5]

    ax.barh(labels, scores, color='steelblue')
    ax.set_xlim(0, 1)
    ax.set_title(q[:30] + '...', fontsize=8)
    ax.set_xlabel('Cosine Similarity')

plt.suptitle('Top-5 Retrieved Chunks per Query')
plt.tight_layout()
plt.show()

## Summary

| Component | Implementation |
|---|---|
| Chunking | Fixed-size with overlap |
| Embedding | `all-MiniLM-L6-v2` (384 dims) |
| Vector store | Numpy cosine similarity |
| Retrieval | Top-k nearest neighbours |
| Generation | Template-based (swap for LLM API) |

**To use with a real LLM**: Replace `simple_answer()` with an Anthropic/OpenAI API call passing `build_prompt()` as the user message. The rest of the pipeline stays identical.